# 01 — Exploración de la API del BCRA

Objetivo: confirmar qué series están disponibles en la API pública del BCRA y bajar las primeras series relevantes para el proyecto (tasas, mora agregada si existiera).

Dos APIs distintas:

- **Principales Variables (v4.0)**: series monetarias/financieras agregadas.
- **Central de Deudores (v1.0)**: consulta por CUIT/CUIL/CDI puntual — **no** trae mora agregada por entidad/cartera.

La mora agregada por entidad y tipo de cartera (consumo/comercial) no tiene API JSON: se publica como Excel en el [Anexo estadístico del Informe sobre Bancos](https://www.bcra.gob.ar/catalogo_de_datos/anexo-estadistico-del-informe-sobre-bancos/). Ese archivo se descarga manualmente y se guarda en `data/raw/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bcra_api import list_monetary_variables, get_monetary_series

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Catálogo de variables monetarias

Si `requests` tira `SSLError` por la cadena de certificados incompleta del lado del BCRA, reintentar con `verify=False` (ver docstring de `src/bcra_api.py`).

In [ ]:
try:
    variables = list_monetary_variables()
except Exception as exc:
    print(f"Fallo con verify=True ({exc}); reintentando con verify=False")
    variables = list_monetary_variables(verify=False)

df_variables = pd.DataFrame(variables)
df_variables.to_csv(RAW_DIR / "bcra_variables_monetarias.csv", index=False)
df_variables.head(20)

## Buscar series relevantes (tasas, cartera irregular)

Filtramos por descripción para encontrar los `idVariable` de interés (tasa de política monetaria, tasas activas, etc.).

In [ ]:
keywords = ["tasa", "cartera", "irregular", "mora"]
mask = df_variables["descripcion"].str.lower().str.contains("|".join(keywords), na=False)
df_variables[mask]

## Descargar una serie de ejemplo

Reemplazar `id_variable` por el que corresponda una vez identificado en la celda anterior.

In [ ]:
id_variable = 6  # placeholder: tasa de política monetaria (TM20) a confirmar contra el catálogo
serie = get_monetary_series(id_variable, desde="2023-01-01")
df_serie = pd.DataFrame(serie)
df_serie.to_csv(RAW_DIR / f"bcra_serie_{id_variable}.csv", index=False)
df_serie.head()

## Próximo paso: mora agregada por entidad

1. Descargar manualmente el Excel del [Anexo estadístico del Informe sobre Bancos](https://www.bcra.gob.ar/catalogo_de_datos/anexo-estadistico-del-informe-sobre-bancos/) y guardarlo en `data/raw/`.
2. Escribir un parser en `src/clean.py` para normalizar las hojas de irregularidad de cartera por grupo de entidades.
3. Continuar en `02_mora_bancos_vs_fintech.ipynb`.